# 04 — Test-set evaluation (one-shot, pre-committed)

Final held-out evaluation on **test (2024-01-13 → 2026-03-28, n=1,141)** with the models retrained on **train + val combined**.

## Decision rule — pre-committed before any test number was inspected

- **Deploy model**: `v3_catboost_full2000_trainval`
- **Sizing config**: ¼-Kelly, no cap
- **Pass threshold**: final bankroll on test ≥ $1.20 (any meaningful growth)
- **Edge threshold (Kelly)**: 3% (PLAN.md §10.1)
- **Edge threshold (flat)**: 5% (STATUS.md leaderboard convention)
- **Market**: Kalshi-like (no-vig prices + 7% fee on winnings)

The full pre-committed script lives at [scripts/test_set_evaluation.py](../scripts/test_set_evaluation.py). This notebook re-runs the same computation for visualization purposes — the decision was already made in the script and is locked.

## Models evaluated

1. **v3_full2000_trainval** — the deploy candidate. v3 scalar Bayesian skill features, CatBoost 2000 iters no early stopping, seed=0, trained on train+val (n=6,028 fights).
2. **v3_full2000_no_skill_corrupted_trainval** — diagnostic counterpart that wraps the above and forces NaN on skill features at inference. Kept to see whether the val-era pattern ("corrupted does suspiciously well at extreme Kelly") replicates on test.

## Note on what "test" means now

After this notebook, the test set is **spent** as a clean held-out resource. Any further modeling iteration cannot use test as a final unbiased evaluation — the next held-out signal will need to come from live events (paper trading or small real-money tranches).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from ufc_pred.backtest.bet_eval import evaluate_bets, evaluate_bets_kelly
from ufc_pred.backtest.metrics import evaluate, market_no_vig_prob_red
from ufc_pred.features.static_v1 import prepare
from ufc_pred.features.skill_v3_pipeline import OUTPUT as SKILL_V3_PARQUET
from ufc_pred.ingest.kaggle_mdabbert import HISTORY_PARQUET
from ufc_pred.utils.time_splits import split

pd.set_option('display.precision', 3)

## Data + predictions

Load test fights, join v3 skill features (already produced by the monthly walk-forward pipeline so they cover test months leak-free), and generate predictions from both models.

In [ ]:
fights = pd.read_parquet(HISTORY_PARQUET)
fights = fights[fights['Winner'].isin(['Red', 'Blue'])].copy()
fights['date'] = pd.to_datetime(fights['date'])

sk = pd.read_parquet(SKILL_V3_PARQUET)
sk['date'] = pd.to_datetime(sk['date'])
fights = fights.merge(
    sk[['date', 'R_fighter', 'B_fighter', 'skill_diff_mean', 'skill_diff_std']],
    on=['date', 'R_fighter', 'B_fighter'], how='left', validate='many_to_one',
)

splits = split(fights)
test = splits.test.reset_index(drop=True)
y_test = (test['Winner'].to_numpy() == 'Red').astype(int)

def predict(model_name):
    payload = joblib.load(ROOT / 'artifacts' / 'models' / f'{model_name}.joblib')
    cols = payload['columns']
    cat = payload.get('cat_features', [])
    X, _, _, _ = prepare(test, augment_symmetry=False, one_hot=False)
    X = X.reindex(columns=cols, fill_value=None)
    for c in cat:
        X[c] = X[c].fillna('__missing__').astype(str)
    return payload['model'].predict_proba(X)[:, 1]

p_v3 = predict('v3_catboost_full2000_trainval')
p_corrupt = predict('v3_full2000_no_skill_corrupted_trainval')

print(f'test n: {len(test)}  date range: {test["date"].min().date()} → {test["date"].max().date()}')
print(f'y_red mean: {y_test.mean():.3f}')
print(f'v3_full2000_trainval:           pred mean {p_v3.mean():.3f}, range [{p_v3.min():.3f}, {p_v3.max():.3f}]')
print(f'v3_full2000_corrupted_trainval: pred mean {p_corrupt.mean():.3f}, range [{p_corrupt.min():.3f}, {p_corrupt.max():.3f}]')

## Predictive metrics on test

Sharpness (log_loss, Brier), calibration (ECE), and accuracy. Market no-vig log_loss on test is included as the wall to climb.

In [ ]:
sub = test.dropna(subset=['R_odds', 'B_odds'])
market = evaluate(
    (sub['Winner'] == 'Red').astype(int).to_numpy(),
    market_no_vig_prob_red(sub), label='test_market',
)
m_v3 = evaluate(y_test, p_v3, label='v3_full2000_trainval')
m_corrupt = evaluate(y_test, p_corrupt, label='v3_full2000_corrupted_trainval')

metrics_df = pd.DataFrame([m_v3, m_corrupt, market])[
    ['label', 'n', 'log_loss', 'brier', 'ece', 'accuracy_argmax']
].rename(columns={'accuracy_argmax': 'accuracy'})
metrics_df

## Flat-stake ROI@5% with bootstrap CI95 — the credibility check

This is the most important table in the notebook. Kelly bankroll numbers compound and can be moved around by a single sequence quirk; flat-stake ROI with bootstrap CI is a stable, comparable estimate of *whether the model has a real edge*.

**What matters:** the CI95 lower bound. If it's > 0, the model demonstrates positive edge that survives bootstrap reweighting of which bets landed in the sample. If it crosses zero, the headline ROI may be lucky.

In [ ]:
def flat(p, label):
    r = evaluate_bets(
        p, y_test, test['R_odds'], test['B_odds'],
        edge_threshold=0.05, fee_rate=0.07, use_no_vig=True,
    )
    return {
        'model': label,
        'n_bets': r.n_bets,
        'roi_pct': r.roi_pct,
        'ci95_low': r.ci95_roi_pct[0],
        'ci95_high': r.ci95_roi_pct[1],
        'mean_ev_pct': r.mean_ev_pct,
        'hit_rate': r.hit_rate,
        'sharpe': r.sharpe,
    }

flat_df = pd.DataFrame([
    flat(p_v3, 'v3_full2000_trainval'),
    flat(p_corrupt, 'v3_full2000_corrupted_trainval'),
])
flat_df

## Kelly sweep grid

Same 4×5 grid as the val notebook: 4 Kelly fractions × 5 caps (1%, 2%, 5%, 10%, no-cap). Bankroll starts at $1, edge threshold = 3%, Kalshi-like market.

In [ ]:
FRACTIONS = [0.10, 0.25, 0.50, 1.00]
CAPS = [0.01, 0.02, 0.05, 0.10, 1.00]
EDGE_THRESHOLD = 0.03
FEE_RATE = 0.07
USE_NO_VIG = True

def sweep(p):
    out = {}
    for f in FRACTIONS:
        for c in CAPS:
            out[(f, c)] = evaluate_bets_kelly(
                p, y_test, test['R_odds'], test['B_odds'],
                edge_threshold=EDGE_THRESHOLD, fee_rate=FEE_RATE,
                use_no_vig=USE_NO_VIG, kelly_fraction=f, max_bet_fraction=c,
                starting_bankroll=1.0,
            )
    return out

grid_v3 = sweep(p_v3)
grid_corrupt = sweep(p_corrupt)
print(f'Sweep done: {len(grid_v3)} configs × 2 models')
print(f'Bets per config (¼-K, no cap): {grid_v3[(0.25, 1.00)]["n_bets"]}')

## Final-bankroll heatmaps

Read with the variance warning in mind: a single cell is a single realization of a stochastic process. A non-monotonic gradient (e.g., ¼-K + no-cap higher than ½-K + no-cap) is a signal of sequence-dependent luck, not a real ordering of strategies.

In [ ]:
def grid_to_df(grid, key):
    data = {c: [grid[(f, c)][key] for f in FRACTIONS] for c in CAPS}
    df = pd.DataFrame(data, index=[f'{int(f*100)}%-K' for f in FRACTIONS])
    df.columns = [(f'{int(c*100)}%' if c < 1 else 'no cap') for c in CAPS]
    df.columns.name = 'per-bet cap'
    df.index.name = 'Kelly fraction'
    return df

bk_v3 = grid_to_df(grid_v3, 'final_bankroll')
bk_corrupt = grid_to_df(grid_corrupt, 'final_bankroll')

print('DEPLOY CANDIDATE — v3_full2000_trainval — final bankroll ($1 → ?)')
display(bk_v3.style
    .format('${:,.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=200, axis=None))

print('\nDIAGNOSTIC — v3_full2000_corrupted_trainval — final bankroll ($1 → ?)')
display(bk_corrupt.style
    .format('${:,.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=200, axis=None))

## Max-drawdown heatmaps

Worst peak-to-trough drop during the simulation. PLAN.md §10.3 says step down to 1/8-K once you hit −25%, so any cell ≥ 25% would have triggered that rule at least once during test.

In [ ]:
dd_v3 = grid_to_df(grid_v3, 'max_drawdown_pct')
dd_corrupt = grid_to_df(grid_corrupt, 'max_drawdown_pct')

print('DEPLOY CANDIDATE — v3_full2000_trainval — max drawdown (%)')
display(dd_v3.style
    .format('{:.1f}%')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=100, axis=None))

print('\nDIAGNOSTIC — v3_full2000_corrupted_trainval — max drawdown (%)')
display(dd_corrupt.style
    .format('{:.1f}%')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=100, axis=None))

## Bankroll over time — all 20 configurations per model

Linear (left) and log (right) scales. Color = Kelly fraction; line style = cap. The pre-committed deploy config (¼-K + no cap) is highlighted with a thicker line.

In [ ]:
FRACTION_COLORS = {0.10: '#a6d96a', 0.25: '#1a9641', 0.50: '#fdae61', 1.00: '#d7191c'}
CAP_LINESTYLES = {0.01: ':', 0.02: '--', 0.05: '-.', 0.10: '-', 1.00: (0, (3, 1, 1, 1))}
CAP_LABELS = {0.01: '1%', 0.02: '2%', 0.05: '5%', 0.10: '10%', 1.00: 'no cap'}
DEPLOY = (0.25, 1.00)

def plot_trajectories(grid, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, scale in zip(axes, ['linear', 'log']):
        for f in FRACTIONS:
            for c in CAPS:
                traj = grid[(f, c)]['trajectory']
                final = grid[(f, c)]['final_bankroll']
                lw = 3.0 if (f, c) == DEPLOY else 1.3
                alpha = 1.0 if (f, c) == DEPLOY else 0.7
                label = f'{int(f*100)}%-K, cap {CAP_LABELS[c]} → ${final:,.2f}'
                if (f, c) == DEPLOY:
                    label += '  ← deploy'
                ax.plot(traj, color=FRACTION_COLORS[f], linestyle=CAP_LINESTYLES[c],
                        linewidth=lw, alpha=alpha, label=label)
        ax.axhline(1.0, color='black', linestyle='-', alpha=0.4, linewidth=0.8)
        ax.axhline(0.75, color='red', linestyle=':', alpha=0.4, linewidth=0.8)
        ax.axhline(1.20, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
        ax.set_xlabel('Bet # (chronological order through test 2024-2026)')
        ax.set_ylabel('Bankroll (starting $1)')
        ax.set_yscale(scale)
        if scale == 'log':
            ax.set_ylim(0.001, 1e6)
        ax.set_title(f'{title} — {scale} scale')
        ax.grid(alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=7, framealpha=0.95, ncol=1)
    plt.tight_layout()
    plt.show()

plot_trajectories(grid_v3, 'DEPLOY CANDIDATE  v3_full2000_trainval')
print('Notes: black solid = $1 break-even.  Red dotted = $0.75 PLAN.md −25% step-down.  Blue dotted = $1.20 pass threshold.')
print('Color = Kelly fraction.  Line style = cap.  Thick green = pre-committed deploy config.')

In [ ]:
plot_trajectories(grid_corrupt, 'DIAGNOSTIC  v3_full2000_corrupted_trainval')

## Verdict — pre-committed decision rule

Below evaluates the locked rule. The verdict was determined by the script before any of the above plots were inspected.

In [ ]:
DEPLOY_THRESHOLD = 1.20
cell = grid_v3[DEPLOY]
passed = cell['final_bankroll'] >= DEPLOY_THRESHOLD

print('=' * 72)
print('DEPLOY DECISION  (rule pre-committed before seeing test results)')
print('=' * 72)
print(f'Model:        v3_catboost_full2000_trainval')
print(f'Config:       25%-Kelly, no cap')
print(f'Threshold:    final bankroll ≥ ${DEPLOY_THRESHOLD:.2f}')
print(f'Result:       $1.00 → ${cell["final_bankroll"]:,.2f}')
print(f'              max DD {cell["max_drawdown_pct"]:.1f}%, n_bets={cell["n_bets"]}')
print(f'Verdict:      {"PASS — deploy criterion met" if passed else "FAIL — do not deploy"}')
print('=' * 72)

## Polymarket scenario — what would the same model earn under Polymarket fees?

The deploy was sized against the Kalshi-like scenario (no-vig + 7% on winnings). Polymarket's UFC market is also no-vig, but the fee structure is different: **3% taker fee** charged on the trade, formula `shares × fee_rate × p × (1−p)`. Per $1 of capital deployed, the effective decimal is `dec / (1 + fee_rate × (1 − 1/dec))`. This is ~5-8× cheaper than Kalshi across the probability range the model bets in.

Re-running the same `v3_catboost_full2000_trainval` model on the same test set with `fee_model="polymarket"` and `fee_rate=0.03` shows what the same predictions would have realized if the deploy had targeted Polymarket instead.


In [ ]:
from ufc_pred.backtest.bet_eval import evaluate_bets_sweep

POLYMARKET_FEE = 0.03
POLYMARKET_THRESHOLD = 0.03  # could safely go lower than Kalshi's 3% but keep parity for comparison

def flat_polymarket(p, label):
    r = evaluate_bets(
        p, y_test, test['R_odds'], test['B_odds'],
        edge_threshold=0.05, fee_rate=POLYMARKET_FEE, use_no_vig=True,
        fee_model='polymarket',
    )
    return {
        'model': label,
        'n_bets': r.n_bets,
        'roi_pct': r.roi_pct,
        'ci95_low': r.ci95_roi_pct[0],
        'ci95_high': r.ci95_roi_pct[1],
        'mean_ev_pct': r.mean_ev_pct,
        'hit_rate': r.hit_rate,
        'sharpe': r.sharpe,
    }

flat_pm_df = pd.DataFrame([
    flat_polymarket(p_v3, 'v3_full2000_trainval'),
    flat_polymarket(p_corrupt, 'v3_full2000_corrupted_trainval'),
])
print('Flat-stake @5% edge — Polymarket scenario (3% taker, no-vig):')
flat_pm_df


In [ ]:
# Side-by-side: same model on same test set, Kalshi vs Polymarket fee model.
compare_df = flat_df.merge(
    flat_pm_df, on='model', suffixes=('_kalshi', '_polymarket'),
)[[
    'model',
    'n_bets_kalshi', 'roi_pct_kalshi', 'ci95_low_kalshi', 'ci95_high_kalshi',
    'n_bets_polymarket', 'roi_pct_polymarket', 'ci95_low_polymarket', 'ci95_high_polymarket',
]]
compare_df


In [ ]:
# Threshold sweep — Polymarket fees admit more bets at low edges. Run the curve.
sweep_pm = evaluate_bets_sweep(
    p_v3, y_test, test['R_odds'], test['B_odds'],
    thresholds=[0.00, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10],
    fee_rate=POLYMARKET_FEE, use_no_vig=True, fee_model='polymarket',
)
sweep_kalshi = evaluate_bets_sweep(
    p_v3, y_test, test['R_odds'], test['B_odds'],
    thresholds=[0.00, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10],
    fee_rate=0.07, use_no_vig=True, fee_model='winnings',
)
print('v3_full2000_trainval — threshold sweep')
print('\nKalshi-like (7% on winnings):')
print(sweep_kalshi[['edge_threshold','n_bets','roi_pct','ci95_low','ci95_high','sharpe']].to_string(index=False))
print('\nPolymarket (3% taker):')
print(sweep_pm[['edge_threshold','n_bets','roi_pct','ci95_low','ci95_high','sharpe']].to_string(index=False))


In [ ]:
# Kelly sweep under polymarket fees — what would the deploy headline look like?
def sweep_kelly_pm(p):
    out = {}
    for f in FRACTIONS:
        for c in CAPS:
            out[(f, c)] = evaluate_bets_kelly(
                p, y_test, test['R_odds'], test['B_odds'],
                edge_threshold=EDGE_THRESHOLD, fee_rate=POLYMARKET_FEE,
                use_no_vig=True, fee_model='polymarket',
                kelly_fraction=f, max_bet_fraction=c,
                starting_bankroll=1.0,
            )
    return out

grid_v3_pm = sweep_kelly_pm(p_v3)
grid_corrupt_pm = sweep_kelly_pm(p_corrupt)

bk_v3_pm = grid_to_df(grid_v3_pm, 'final_bankroll')
bk_corrupt_pm = grid_to_df(grid_corrupt_pm, 'final_bankroll')

print('POLYMARKET — v3_full2000_trainval — final bankroll ($1 → ?)')
display(bk_v3_pm.style
    .format('${:,.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=200, axis=None))

print('\nPOLYMARKET — v3_full2000_corrupted_trainval — final bankroll ($1 → ?)')
display(bk_corrupt_pm.style
    .format('${:,.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=200, axis=None))


In [ ]:
# Direct headline comparison at the pre-committed deploy config (¼-K, no cap).
cell_kalshi = grid_v3[DEPLOY]
cell_pm = grid_v3_pm[DEPLOY]
cell_corrupt_kalshi = grid_corrupt[DEPLOY]
cell_corrupt_pm = grid_corrupt_pm[DEPLOY]

summary_df = pd.DataFrame([
    {'scenario': 'Kalshi (7% on winnings)', 'model': 'v3_real',
     'final_bankroll': cell_kalshi['final_bankroll'], 'max_dd_pct': cell_kalshi['max_drawdown_pct'],
     'n_bets': cell_kalshi['n_bets']},
    {'scenario': 'Polymarket (3% taker)', 'model': 'v3_real',
     'final_bankroll': cell_pm['final_bankroll'], 'max_dd_pct': cell_pm['max_drawdown_pct'],
     'n_bets': cell_pm['n_bets']},
    {'scenario': 'Kalshi (7% on winnings)', 'model': 'v3_corrupted',
     'final_bankroll': cell_corrupt_kalshi['final_bankroll'], 'max_dd_pct': cell_corrupt_kalshi['max_drawdown_pct'],
     'n_bets': cell_corrupt_kalshi['n_bets']},
    {'scenario': 'Polymarket (3% taker)', 'model': 'v3_corrupted',
     'final_bankroll': cell_corrupt_pm['final_bankroll'], 'max_dd_pct': cell_corrupt_pm['max_drawdown_pct'],
     'n_bets': cell_corrupt_pm['n_bets']},
])
print('Deploy config (¼-K, no cap) on test — $1 → ?')
summary_df.style.format({'final_bankroll': '${:,.2f}', 'max_dd_pct': '{:.1f}%'})


**Reading the comparison:** Polymarket's fee structure is materially cheaper than Kalshi's, so the same predictions earn more per bet. Whether the headline dollar figure goes up or down at the no-cap deploy config depends on how the bet sequencing compounds — a fee change reshuffles which fights clear the edge threshold and how big each Kelly stake is, which can amplify or dampen sequence luck. **The relevant signal is the flat-stake ROI and threshold-sweep ROI, not the no-cap compounded bankroll** — those are sequence-luck-amplified per [DEPLOY.md §5.3](../DEPLOY.md). Use this section as evidence that switching venue is positive-EV, not as a re-do of the deploy decision.


## Did the val story replicate on test?

A side-by-side check. The val numbers come from notebook 03 (train-only models) and test from this notebook (train+val models). Not identical setups but the architectural family and Kelly mechanics are the same.

In [ ]:
# Val numbers from notebook 03 (hardcoded reference for comparison).
val_reference = {
    ('v3', 0.25, 0.02):  1.90,
    ('v3', 0.25, 0.05):  4.29,
    ('v3', 0.25, 1.00): 45.66,
    ('corrupt', 0.25, 0.02):   1.76,
    ('corrupt', 0.25, 1.00): 103.13,
}

rows = []
for model_label, grid in [('v3', grid_v3), ('corrupt', grid_corrupt)]:
    for cap in [0.02, 0.05, 1.00]:
        key = (model_label, 0.25, cap)
        if key not in val_reference:
            continue
        rows.append({
            'model': model_label,
            'config': f'¼-K, {"no cap" if cap >= 1 else f"{int(cap*100)}%"}',
            'val ($1 → ?)': f'${val_reference[key]:,.2f}',
            'test ($1 → ?)': f'${grid[(0.25, cap)]["final_bankroll"]:,.2f}',
        })
pd.DataFrame(rows)

## Verification — when did the extreme drawdowns happen?

Some grid cells show very high max drawdowns (88% for ¼-K + no-cap on the real model; 93.5% for the same on corrupted; 99.9% at ½-K + no-cap on corrupted) and still end at large final bankrolls. That deserves a sanity check.

The `max_drawdown_pct` from `evaluate_bets_kelly` is the running max of `(peak − bankroll) / peak`, where `peak` is the highest bankroll seen up to that point in the trajectory. A 99.9% DD with a positive final means: bankroll touched 0.1% of its then-peak at some bet, and then later climbed back (not necessarily above the original peak — the running max retains the worst observed value).

Below we re-implement the bankroll simulation independently from `bet_eval.py`, verify the numbers match the library, locate the exact bet at which peak and trough occurred, and visualize each case.

In [ ]:
from ufc_pred.backtest.bet_eval import american_to_decimal
from ufc_pred.backtest.metrics import american_to_implied_prob


def simulate_with_logging(p, y, R_odds, B_odds,
                          kelly_fraction, max_bet_fraction,
                          edge_threshold=0.03, fee_rate=0.07, use_no_vig=True):
    """Independent re-implementation. Returns (per-bet log, max_dd_pct)."""
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=int)

    dec_R = american_to_decimal(R_odds)
    dec_B = american_to_decimal(B_odds)
    valid = ~(np.isnan(dec_R) | np.isnan(dec_B))

    if use_no_vig:
        p_r_imp = american_to_implied_prob(R_odds)
        p_b_imp = american_to_implied_prob(B_odds)
        tot = p_r_imp + p_b_imp
        dec_R = 1.0 / (p_r_imp / tot)
        dec_B = 1.0 / (p_b_imp / tot)

    eff_R = 1.0 + (1.0 - fee_rate) * (dec_R - 1.0)
    eff_B = 1.0 + (1.0 - fee_rate) * (dec_B - 1.0)
    p_R, p_B = p, 1.0 - p
    ev_R = p_R * eff_R - 1.0
    ev_B = p_B * eff_B - 1.0
    bet_red = ev_R >= ev_B
    chosen_ev = np.where(bet_red, ev_R, ev_B)
    chosen_dec = np.where(bet_red, eff_R, eff_B)
    chosen_p = np.where(bet_red, p_R, p_B)
    bets_mask = valid & (chosen_ev > edge_threshold)
    won_full = np.where(bet_red, y == 1, y == 0)

    bankroll, peak, max_dd = 1.0, 1.0, 0.0
    rows, bet_idx = [], 0
    for i in range(len(p_R)):
        if not bets_mask[i]:
            continue
        b = chosen_dec[i] - 1.0
        pi, qi = chosen_p[i], 1.0 - chosen_p[i]
        full_kelly = (b * pi - qi) / b
        if full_kelly <= 0:
            continue
        stake_frac = min(kelly_fraction * full_kelly, max_bet_fraction)
        stake = bankroll * stake_frac
        if won_full[i]:
            bankroll += stake * (chosen_dec[i] - 1.0)
        else:
            bankroll -= stake
        peak = max(peak, bankroll)
        dd = (peak - bankroll) / peak
        max_dd = max(max_dd, dd)
        rows.append({
            "bet_idx": bet_idx, "fight_idx": i,
            "date": test["date"].iloc[i].date(),
            "side": "R" if bet_red[i] else "B",
            "p_chosen": chosen_p[i],
            "decimal_odds": chosen_dec[i],
            "stake_frac": stake_frac,
            "stake": stake,
            "won": int(won_full[i]),
            "bankroll_after": bankroll,
            "peak_so_far": peak,
            "drawdown_pct": dd * 100,
        })
        bet_idx += 1
    return pd.DataFrame(rows), max_dd * 100


def verify_case(label, p, kelly_fraction, max_bet_fraction):
    log, max_dd_ours = simulate_with_logging(
        p, y_test, test["R_odds"], test["B_odds"],
        kelly_fraction=kelly_fraction, max_bet_fraction=max_bet_fraction,
    )
    lib = evaluate_bets_kelly(
        p, y_test, test["R_odds"], test["B_odds"],
        edge_threshold=0.03, fee_rate=0.07, use_no_vig=True,
        kelly_fraction=kelly_fraction, max_bet_fraction=max_bet_fraction,
    )
    peak_bet = log["peak_so_far"].idxmax()
    dd_bet = log["drawdown_pct"].idxmax()
    print(f"=== {label}  ({int(kelly_fraction*100)}%-K, "
          f"{'no cap' if max_bet_fraction >= 1 else f'{int(max_bet_fraction*100)}% cap'}) ===")
    print(f"  library:  final ${lib['final_bankroll']:.4g}   max DD {lib['max_drawdown_pct']:.3f}%")
    print(f"  ours:     final ${log['bankroll_after'].iloc[-1]:.4g}   max DD {max_dd_ours:.3f}%")
    print(f"  match:    final={np.isclose(lib['final_bankroll'], log['bankroll_after'].iloc[-1], rtol=1e-9)}"
          f"   maxDD={np.isclose(lib['max_drawdown_pct'], max_dd_ours, atol=1e-6)}")
    print(f"  overall peak     = ${log['peak_so_far'].iloc[peak_bet]:.4g} "
          f"at bet #{peak_bet}/{len(log)}  ({log['date'].iloc[peak_bet]})")
    print(f"  worst-DD trough  = ${log['bankroll_after'].iloc[dd_bet]:.4g} "
          f"(peak-so-far ${log['peak_so_far'].iloc[dd_bet]:.4g}) "
          f"at bet #{dd_bet}/{len(log)}  ({log['date'].iloc[dd_bet]})")
    print(f"  final bankroll   = ${log['bankroll_after'].iloc[-1]:.4g}")
    print()
    return log


CASES = [
    ("v3_real DEPLOY-B",       p_v3,      0.25, 1.00),
    ("v3_real ½-K + no cap",   p_v3,      0.50, 1.00),
    ("v3_real full-K + no cap (RUIN)", p_v3, 1.00, 1.00),
    ("corrupted DEPLOY-C",     p_corrupt, 0.25, 1.00),
    ("corrupted ½-K + no cap", p_corrupt, 0.50, 1.00),
    ("v3_real DEPLOY-A",       p_v3,      0.10, 0.10),
]
LOGS = {(label, f, c): verify_case(label, p, f, c) for label, p, f, c in CASES}


In [ ]:
def plot_case(log, title):
    fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

    ax = axes[0]
    ax.plot(log["bet_idx"], log["bankroll_after"], color="#1a9641",
            linewidth=1.5, label="bankroll")
    ax.plot(log["bet_idx"], log["peak_so_far"], color="#fdae61",
            linewidth=1.5, linestyle="--", alpha=0.85, label="running peak")
    peak_idx = log["peak_so_far"].idxmax()
    dd_idx = log["drawdown_pct"].idxmax()
    ax.scatter([log["bet_idx"].iloc[peak_idx]],
               [log["peak_so_far"].iloc[peak_idx]],
               color="black", s=70, zorder=5,
               label=f"overall peak (${log['peak_so_far'].iloc[peak_idx]:.4g})")
    ax.scatter([log["bet_idx"].iloc[dd_idx]],
               [log["bankroll_after"].iloc[dd_idx]],
               color="red", s=70, marker="v", zorder=5,
               label=f"worst-DD trough (${log['bankroll_after'].iloc[dd_idx]:.4g}, "
                     f"{log['drawdown_pct'].iloc[dd_idx]:.1f}%)")
    ax.scatter([log["bet_idx"].iloc[-1]],
               [log["bankroll_after"].iloc[-1]],
               color="blue", s=70, marker="s", zorder=5,
               label=f"final (${log['bankroll_after'].iloc[-1]:.4g})")
    # Y-axis: log only if range spans > 1 order of magnitude.
    bk_min = max(log["bankroll_after"].min(), 1e-6)
    bk_max = max(log["peak_so_far"].max(), bk_min * 10)
    if bk_max / bk_min > 10:
        ax.set_yscale("log")
        ax.set_ylabel("Bankroll ($, log)")
    else:
        ax.set_ylabel("Bankroll ($)")
    ax.set_title(f"{title} — bankroll + running peak")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3, which="both")

    ax = axes[1]
    ax.fill_between(log["bet_idx"], 0, log["drawdown_pct"], color="#d7191c", alpha=0.4)
    ax.plot(log["bet_idx"], log["drawdown_pct"], color="#d7191c", linewidth=1.2)
    ax.scatter([log["bet_idx"].iloc[dd_idx]],
               [log["drawdown_pct"].iloc[dd_idx]],
               color="red", s=70, marker="v", zorder=5)
    ax.axhline(50, color="gray", linestyle=":", alpha=0.5)
    ax.axhline(90, color="gray", linestyle=":", alpha=0.5)
    ax.set_ylim(0, 102)
    ax.set_ylabel("Drawdown from running peak (%)")
    ax.set_xlabel("Bet # (chronological)")
    ax.set_title("Drawdown trajectory")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


for (label, f, c), log in LOGS.items():
    plot_case(log, f"{label}  ({int(f*100)}%-K, "
                   f"{'no cap' if c >= 1 else f'{int(c*100)}% cap'})")


### What the verification shows

1. **Library numbers match the independent reimplementation exactly** (final bankroll and max DD to numerical precision). The `evaluate_bets_kelly` code is correct.

2. **Big drawdowns are real and happen mid-arc, not at the end.** The "running peak" line in each plot shows that for the no-cap cases, peak bankroll is often reached *during* the trajectory (not at the end), and a single very large losing bet collapses the bankroll to a small fraction of that peak. Then a subsequent winning sequence climbs back — sometimes past the old peak, sometimes not (in which case the recorded max DD is preserved even though the final bankroll is much higher than the trough).

3. **The 99.9% case (corrupted ½-K + no cap) is genuine.** The peak is reached, a single max-confidence loss takes ~99% of the bankroll in one bet (Kelly with p≈0.99 stakes nearly all the bankroll), and subsequent wins compound back from the tiny remainder. The final value being $147k from a bankroll that briefly held $1k or less is mathematically consistent with how Kelly compounds — but this is exactly the kind of single-sample-luck path the deploy plan acknowledges in the risk section.

4. **The full-Kelly + no-cap RUIN case shows the other tail.** Same mechanism: a single max-confidence loss wipes the bankroll out, and there's nothing left to compound back from. ¼-K is the difference between "lose 25% of the bankroll on one bad bet and survive" and "lose everything."

5. **Account A (10%-K + 10% cap) shows the well-behaved profile.** Drawdown stays bounded (≤62% on test), bankroll line is much smoother. This is what "Kelly with a safety cap" actually looks like vs the no-cap accounts.

The takeaway for the deployment: the headline bankroll numbers for no-cap accounts are real but path-dependent in an extreme way. The compounding gain requires surviving (or just barely surviving) one or more single-bet wipeouts. Live, that's a bet where $300 might briefly be $3 before recovering — a session you have to commit to not bailing on for the strategy's expected outcome to be realized.

## Per-fight-night results for the three deploy accounts

UFC schedules fights into events (fight nights), each on a single date. Above we plotted bankroll vs bet number, which compresses bets within the same night into the same x-position. Here we re-do the same simulation but starting each account at **$300** and aggregate by date, so you can see:

- How much each account would have made/lost on each event during the 2-year test window
- The cumulative bankroll trajectory at fight-night granularity
- Which specific events drove the headline numbers

**The three accounts (from [DEPLOY.md](../DEPLOY.md)):**
- **A**: $300, 10%-K + 10% cap, v3_full2000_trainval
- **B**: $300, ¼-K + no cap, v3_full2000_trainval
- **C**: $300, ¼-K + no cap, v3_full2000_corrupted_trainval

In [ ]:
ACCOUNTS = [
    ("A", "v3_real",   p_v3,      0.10, 0.10, "#1a9641"),
    ("B", "v3_real",   p_v3,      0.25, 1.00, "#1f78b4"),
    ("C", "corrupted", p_corrupt, 0.25, 1.00, "#d7191c"),
]
START = 300.0


def simulate_with_dates(p, kelly_fraction, max_bet_fraction, start=START):
    """Like simulate_with_logging but starts at $start and tags each bet with its date."""
    log, _ = simulate_with_logging(
        p, y_test, test["R_odds"], test["B_odds"],
        kelly_fraction=kelly_fraction, max_bet_fraction=max_bet_fraction,
    )
    # log["bankroll_after"] is built from start=$1. Rescale to start=$300.
    log = log.copy()
    log["bankroll_after"] = log["bankroll_after"] * start
    log["stake"] = log["stake"] * start
    log["peak_so_far"] = log["peak_so_far"] * start
    return log


def aggregate_by_night(log, start=START):
    """Per-night summary: n_bets, n_wins, P&L that night, bankroll at end-of-night."""
    g = log.groupby("date", sort=True)
    nights = g.agg(
        n_bets=("bet_idx", "size"),
        n_wins=("won", "sum"),
        stake_total=("stake", "sum"),
        bankroll_end=("bankroll_after", "last"),
    ).reset_index()
    # P&L per night = bankroll_end[i] - bankroll_end[i-1] (or - start for i=0).
    prev = np.concatenate([[start], nights["bankroll_end"].iloc[:-1].to_numpy()])
    nights["bankroll_start"] = prev
    nights["pnl"] = nights["bankroll_end"] - prev
    nights["pnl_pct"] = nights["pnl"] / prev * 100
    return nights


nightly = {}
for tag, model_label, p, f, c, color in ACCOUNTS:
    log = simulate_with_dates(p, f, c)
    nightly[tag] = aggregate_by_night(log)
    print(f"Account {tag} ({model_label}, {int(f*100)}%-K, "
          f"{'no cap' if c >= 1 else f'{int(c*100)}% cap'}): "
          f"{len(nightly[tag])} fight nights, "
          f"${START:.0f} → ${nightly[tag]['bankroll_end'].iloc[-1]:,.2f}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top: linear scale.
ax = axes[0]
for tag, _, _, f, c, color in ACCOUNTS:
    n = nightly[tag]
    ax.plot(n["date"], n["bankroll_end"], color=color, linewidth=1.8,
            label=f"Account {tag}: ${START:.0f} → ${n['bankroll_end'].iloc[-1]:,.2f}")
ax.axhline(START, color="black", linestyle="-", alpha=0.4, linewidth=0.8,
           label=f"${START:.0f} starting")
ax.set_ylabel("Bankroll ($)")
ax.set_title("Bankroll over time — end of each fight night (linear)")
ax.legend(loc="upper left", fontsize=10)
ax.grid(alpha=0.3)

# Bottom: log scale.
ax = axes[1]
for tag, _, _, f, c, color in ACCOUNTS:
    n = nightly[tag]
    ax.plot(n["date"], n["bankroll_end"], color=color, linewidth=1.8,
            label=f"Account {tag}")
ax.axhline(START, color="black", linestyle="-", alpha=0.4, linewidth=0.8)
ax.set_yscale("log")
ax.set_ylabel("Bankroll ($, log scale)")
ax.set_xlabel("Date (fight night)")
ax.set_title("Bankroll over time — end of each fight night (log)")
ax.legend(loc="lower right", fontsize=10)
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

# Per-night P&L bars (overlapping)
fig, ax = plt.subplots(figsize=(14, 5))
width = 6  # days
for offset, (tag, _, _, f, c, color) in enumerate(ACCOUNTS):
    n = nightly[tag]
    ax.bar(n["date"] + pd.Timedelta(days=offset * 2 - 2),
           n["pnl"], width=width, color=color, alpha=0.6,
           label=f"Account {tag} per-night P&L")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("P&L on that night ($)")
ax.set_xlabel("Fight night")
ax.set_title("Per-fight-night P&L for each account (log absolute Y)")
ax.set_yscale("symlog", linthresh=10)
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


In [ ]:
def merged_table():
    """One row per fight night, columns per account showing end-of-night bankroll and P&L."""
    base_dates = nightly["A"][["date", "n_bets"]].copy()
    for tag, _, _, f, c, color in ACCOUNTS:
        n = nightly[tag][["date", "bankroll_end", "pnl"]].rename(
            columns={"bankroll_end": f"{tag}_end", "pnl": f"{tag}_pnl"})
        base_dates = base_dates.merge(n, on="date", how="left")
    return base_dates


tbl = merged_table()
print(f"Total fight nights with bets: {len(tbl)}")
print(f"Date range: {tbl['date'].min()} → {tbl['date'].max()}")
print()
print("=== First 10 fight nights ===")
print(tbl.head(10).to_string(index=False,
    formatters={
        "n_bets": "{:.0f}".format,
        "A_end": "${:,.2f}".format, "A_pnl": "${:+,.2f}".format,
        "B_end": "${:,.2f}".format, "B_pnl": "${:+,.2f}".format,
        "C_end": "${:,.2f}".format, "C_pnl": "${:+,.2f}".format,
    }))
print()
print("=== Last 10 fight nights ===")
print(tbl.tail(10).to_string(index=False,
    formatters={
        "n_bets": "{:.0f}".format,
        "A_end": "${:,.2f}".format, "A_pnl": "${:+,.2f}".format,
        "B_end": "${:,.2f}".format, "B_pnl": "${:+,.2f}".format,
        "C_end": "${:,.2f}".format, "C_pnl": "${:+,.2f}".format,
    }))


In [ ]:
# Top 10 best and worst nights per account by absolute P&L.
def biggest_moves(tag, n=5):
    n_df = nightly[tag][["date", "n_bets", "n_wins", "bankroll_start", "pnl", "pnl_pct", "bankroll_end"]].copy()
    best = n_df.nlargest(n, "pnl").assign(rank="best")
    worst = n_df.nsmallest(n, "pnl").assign(rank="worst")
    return pd.concat([best, worst], ignore_index=True).assign(account=tag)


moves = pd.concat([biggest_moves(tag, n=5) for tag, _, _, _, _, _ in ACCOUNTS], ignore_index=True)
moves = moves[["account", "rank", "date", "n_bets", "n_wins", "bankroll_start", "pnl", "pnl_pct", "bankroll_end"]]
print("=== Top 5 best + 5 worst nights per account ===")
print(moves.to_string(index=False,
    formatters={
        "n_bets": "{:.0f}".format, "n_wins": "{:.0f}".format,
        "bankroll_start": "${:,.2f}".format,
        "pnl": "${:+,.2f}".format,
        "pnl_pct": "{:+.1f}%".format,
        "bankroll_end": "${:,.2f}".format,
    }))


### What the per-night view shows

1. **The headline final bankrolls assume you survive specific big nights.** The "worst nights" table shows individual events where Account B and C lost 80-95% of bankroll in a single night. If you'd panic-sold mid-event or skipped that night for any reason, the recovery wouldn't have happened.

2. **Most nights are small moves; a handful drive the total.** Looking at the "best 5" rows for each account, a few big winning nights generate most of the bankroll growth. This is geometric compounding amplifying high-confidence bets when they happen to hit on a single event.

3. **Account A's nightly P&L is much smoother.** Because of the 10% cap, no single fight can stake more than 10% of bankroll, so no single night swings the account dramatically. This is what a "safer" deployment actually looks like in practice — narrower nightly P&L distribution, lower final number.

4. **The bars chart (symlog Y) makes the asymmetry visible.** Accounts B and C have rare but very large gain nights (and one or two large loss nights). Account A's bars stay close to zero across the timeline. Same model edge, three completely different risk profiles.

## Takeaways

1. **Pre-committed rule: PASS.** v3_full2000_trainval at ¼-Kelly + no cap turned $1 into $3,500.95 on test. Threshold was $1.20 (any meaningful growth). Decision is binding.

2. **The credible signal is the flat-stake ROI with positive CI lower bound.** ROI@5% = +10.88%, CI95 = (+2.75%, +19.39%). The lower bound being > 0 means the model demonstrates positive edge that survives bootstrap reweighting of the 846 bets. This is the strongest evidence of real edge the project has produced, and is meaningfully stronger than val (which had CI95 lower bound of −3.6%, including zero).

3. **The $3,500 headline is not the expected outcome.** Read across the no-cap column for the deploy model: 10%-K → $103, **¼-K → $3,501**, ½-K → $248, 100%-K → $0. The non-monotonic pattern (½-K should compound *more* than ¼-K under a stable edge but here it compounds 14× less) is the signature of bet-sequence luck. A different ordering of the 884 fights could move the ¼-K + no-cap result anywhere from ~$100 to ruin. The CI on the underlying flat-stake ROI is the better anchor for expected outcome.

4. **88% max drawdown is real.** At some point during the 2-year test window, the bankroll was at $0.12 on the dollar before recovering. Live, that's a session where PLAN.md §10.3's −25% step-down rule would have been hit multiple times. If you're going to deploy at no-cap, do it with money you can stomach watching go to ~12% of starting before it recovers (if it recovers — that's one sequence out of many possible).

5. **Cap is still the binding constraint, even on test.** At ¼-K: 2% cap → $5.26; 5% cap → $25.58; 10% cap → $144.91; no-cap → $3,501. The variance scales much faster than the median outcome. For someone deploying small capital who values *reliable* multiplication over *tail-lottery* multiplication, ¼-K + 5% cap delivers $25.58× growth with 57.5% drawdown — a less dramatic shape than no-cap.

6. **Corrupted model continues to behave strangely at extreme configs.** At ¼-K + no-cap on test it reaches $162,764, vastly more than the real model. Same pattern as val: less-confident predictions stake smaller and dodge wipeouts in this particular sequence. This is not evidence the corrupted model is better — it's further evidence that no-cap numbers are sequence-luck-dominated. At PLAN.md-default sizing (¼-K + 2% cap), corrupted = $4.32 vs real = $5.26 — real wins where the noise is dampened.

7. **log_loss improved.** Test log_loss = 0.6226 vs train-only v3_full2000 on val = 0.6388. Retraining on train+val helped predictive sharpness, as expected. Test market log_loss = 0.5816 remains the wall to climb.

8. **Test is spent.** Any future modeling iteration cannot rely on test for a clean held-out signal. The next held-out evaluation comes from live events (paper trading or small real-money tranches). The deploy pathway is now: stake a small real-money tranche at the committed sizing, treat ongoing P&L as the new held-out evaluation, and reserve the right to step down per PLAN.md §10.3 if drawdown exceeds −25%.